In [1]:
"""
Full Multi-Asset Data Downloader — run this LOCALLY
====================================================
Downloads 163 tickers across 6 asset classes via yfinance.
Requires: pip install yfinance pyarrow pandas numpy

Output (same directory as this script):
  multiasset_prices.parquet   — wide: date × ticker (adjusted close / last)
  multiasset_returns.parquet  — daily log returns
  multiasset_metadata.parquet — ticker metadata
"""

import yfinance as yf
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from datetime import datetime
import warnings, time, os
warnings.filterwarnings("ignore")

UNIVERSE = {
    "equities": {
        "SPY":"SPDR S&P 500 ETF","QQQ":"Invesco Nasdaq-100 ETF","IWM":"iShares Russell 2000 ETF",
        "DIA":"SPDR Dow Jones ETF","VTI":"Vanguard Total Stock Market ETF","VT":"Vanguard Total World ETF",
        "EFA":"iShares MSCI EAFE ETF","EEM":"iShares MSCI Emerging Markets ETF","VEA":"Vanguard Developed Markets ETF",
        "VWO":"Vanguard Emerging Markets ETF","EWJ":"iShares MSCI Japan ETF","EWZ":"iShares MSCI Brazil ETF",
        "FXI":"iShares China Large-Cap ETF","INDA":"iShares MSCI India ETF","EWY":"iShares MSCI South Korea ETF",
        "XLK":"Technology SPDR","XLF":"Financial SPDR","XLE":"Energy SPDR","XLV":"Health Care SPDR",
        "XLI":"Industrial SPDR","XLP":"Consumer Staples SPDR","XLY":"Consumer Discretionary SPDR",
        "XLU":"Utilities SPDR","XLRE":"Real Estate SPDR","XLB":"Materials SPDR","XLC":"Comms SPDR",
        "VTV":"Vanguard Value ETF","VUG":"Vanguard Growth ETF","MTUM":"iShares Momentum Factor ETF",
        "USMV":"iShares Min Vol Factor ETF","QUAL":"iShares Quality Factor ETF","SIZE":"iShares Size Factor ETF",
        "SSO":"ProShares Ultra S&P500 2x","SDS":"ProShares UltraShort S&P500 -2x","TQQQ":"ProShares UltraQQQ 3x",
        "ARKK":"ARK Innovation ETF","ICLN":"iShares Global Clean Energy ETF",
        "ITB":"iShares US Home Construction ETF","XBI":"SPDR S&P Biotech ETF","SOXX":"iShares Semiconductor ETF",
        "AAPL":"Apple","MSFT":"Microsoft","NVDA":"NVIDIA","AMZN":"Amazon","GOOGL":"Alphabet",
        "META":"Meta Platforms","TSLA":"Tesla","JPM":"JPMorgan","BRK-B":"Berkshire B","XOM":"ExxonMobil",
    },
    "crypto": {
        "BTC-USD":"Bitcoin","ETH-USD":"Ethereum","SOL-USD":"Solana","BNB-USD":"BNB","XRP-USD":"XRP",
        "ADA-USD":"Cardano","AVAX-USD":"Avalanche","DOGE-USD":"Dogecoin","DOT-USD":"Polkadot",
        "LINK-USD":"Chainlink","MATIC-USD":"Polygon","LTC-USD":"Litecoin","BCH-USD":"Bitcoin Cash",
        "UNI-USD":"Uniswap","ATOM-USD":"Cosmos",
        "IBIT":"iShares Bitcoin Trust ETF","FBTC":"Fidelity Bitcoin ETF",
        "GBTC":"Grayscale Bitcoin Trust","ETHE":"Grayscale Ethereum Trust","BITB":"Bitwise Bitcoin ETF",
    },
    "volatility": {
        "^VIX":"CBOE VIX","^VXN":"Nasdaq-100 VIX","^RVX":"Russell 2000 VIX","^OVX":"Crude Oil VIX",
        "^GVZ":"Gold VIX","^EVZ":"EuroCurrency VIX","^VVIX":"VIX of VIX","^SKEW":"CBOE Skew",
        "UVXY":"ProShares Ultra VIX STF","SVXY":"ProShares Short VIX STF",
        "VXX":"iPath S&P 500 VIX STF","VIXY":"ProShares VIX STF","VIXM":"ProShares VIX MTF",
    },
    "commodities": {
        "GLD":"SPDR Gold","IAU":"iShares Gold","SLV":"iShares Silver","PPLT":"Aberdeen Platinum","PALL":"Aberdeen Palladium",
        "USO":"US Oil Fund (WTI)","BNO":"US Brent Oil Fund","UNG":"US Natural Gas Fund","UGA":"US Gasoline Fund",
        "CORN":"Teucrium Corn","WEAT":"Teucrium Wheat","SOYB":"Teucrium Soybean","CANE":"Teucrium Sugar",
        "NIB":"iPath Cocoa","JO":"iPath Coffee",
        "DJP":"iPath Bloomberg Commodity","PDBC":"Invesco Diversified Commodity","DBC":"Invesco DB Commodity","GSG":"iShares GSCI",
        "CPER":"US Copper Index Fund","DBB":"Invesco DB Base Metals",
        "GC=F":"Gold Futures","SI=F":"Silver Futures","CL=F":"WTI Crude Futures","BZ=F":"Brent Crude Futures",
        "NG=F":"Natural Gas Futures","HG=F":"Copper Futures","ZC=F":"Corn Futures","ZW=F":"Wheat Futures","ZS=F":"Soybean Futures",
    },
    "rates": {
        "SHY":"iShares 1-3Y Treasury","IEI":"iShares 3-7Y Treasury","IEF":"iShares 7-10Y Treasury",
        "TLH":"iShares 10-20Y Treasury","TLT":"iShares 20Y+ Treasury","ZROZ":"PIMCO 25Y+ Zero Coupon",
        "EDV":"Vanguard Extended Duration Treasury","TIP":"iShares TIPS","SCHP":"Schwab TIPS",
        "LQD":"iShares IG Corp Bond","HYG":"iShares HY Corp Bond","JNK":"SPDR HY Bond",
        "EMB":"iShares EM Bond","AGG":"iShares Core US Aggregate","BND":"Vanguard Total Bond",
        "VCSH":"Vanguard ST Corp Bond","VCIT":"Vanguard IT Corp Bond","VCLT":"Vanguard LT Corp Bond",
        "MUB":"iShares National Muni","MBB":"iShares MBS",
        "^TNX":"10Y Treasury Yield","^TYX":"30Y Treasury Yield","^FVX":"5Y Treasury Yield","^IRX":"13W T-Bill",
        "TMF":"Direxion 20Y+ Treasury Bull 3x","TBT":"ProShares UltraShort 20Y+","TBF":"ProShares Short 20Y+",
    },
    "fx": {
        "EURUSD=X":"EUR/USD","GBPUSD=X":"GBP/USD","JPYUSD=X":"JPY/USD","CHFUSD=X":"CHF/USD",
        "CADUSD=X":"CAD/USD","AUDUSD=X":"AUD/USD","NZDUSD=X":"NZD/USD",
        "SGDUSD=X":"SGD/USD","CNYUSD=X":"CNY/USD","INRUSD=X":"INR/USD","BRLUSD=X":"BRL/USD",
        "MXNUSD=X":"MXN/USD","ZARUSD=X":"ZAR/USD","TRYYUSD=X":"TRY/USD","KRWUSD=X":"KRW/USD","THBUSD=X":"THB/USD",
        "UUP":"Invesco DB USD Bullish","UDN":"Invesco DB USD Bearish",
        "EURJPY=X":"EUR/JPY","EURGBP=X":"EUR/GBP","GBPJPY=X":"GBP/JPY","AUDJPY=X":"AUD/JPY","EURCHF=X":"EUR/CHF",
    },
}

all_prices = {}
metadata_rows = []

for asset_class, tickers in UNIVERSE.items():
    ticker_list = list(tickers.keys())
    print(f"\n{'─'*60}\n  {asset_class.upper()}  ({len(ticker_list)} tickers)\n{'─'*60}")
    for i in range(0, len(ticker_list), 20):
        batch = ticker_list[i:i+20]
        try:
            raw = yf.download(batch, period="max", interval="1d", auto_adjust=True,
                              progress=False, threads=True)
        except Exception as e:
            print(f"  [batch error] {e}"); continue
        if raw.empty: continue
        close = raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw[["Close"]].rename(columns={"Close": batch[0]})
        for ticker in batch:
            name = tickers[ticker]
            if ticker in close.columns:
                s = close[ticker].dropna()
                if len(s) > 10:
                    all_prices[ticker] = s
                    metadata_rows.append(dict(
                        ticker=ticker, asset_class=asset_class, full_name=name,
                        start_date=s.index.min().date(), end_date=s.index.max().date(), n_obs=len(s)
                    ))
                    print(f"  ✓ {ticker:<14} {name[:38]:<40} {len(s):>5} obs  [{s.index.min().date()} → {s.index.max().date()}]")
        time.sleep(0.3)

prices = pd.DataFrame(all_prices).sort_index()
prices.index.name = "date"
log_returns = np.log(prices / prices.shift(1))
metadata = pd.DataFrame(metadata_rows)

try:
    out = os.path.dirname(os.path.abspath(__file__))
except NameError:
    out = os.getcwd()  # running in Jupyter — save to current working directory
prices.to_parquet(f"{out}/multiasset_prices.parquet",   engine="pyarrow", compression="snappy")
log_returns.to_parquet(f"{out}/multiasset_returns.parquet", engine="pyarrow", compression="snappy")
metadata.to_parquet(f"{out}/multiasset_metadata.parquet",  engine="pyarrow", compression="snappy", index=False)

print(f"\n{'='*60}")
print(f"  Downloaded {len(prices.columns)} tickers")
print(f"  Date range: {prices.index.min().date()} → {prices.index.max().date()}")
print(f"  Saved to: {out}")
print(f"{'='*60}")


────────────────────────────────────────────────────────────
  EQUITIES  (50 tickers)
────────────────────────────────────────────────────────────
  ✓ SPY            SPDR S&P 500 ETF                          8392 obs  [1993-01-29 → 2026-06-02]
  ✓ QQQ            Invesco Nasdaq-100 ETF                    6850 obs  [1999-03-10 → 2026-06-02]
  ✓ IWM            iShares Russell 2000 ETF                  6542 obs  [2000-05-26 → 2026-06-02]
  ✓ DIA            SPDR Dow Jones ETF                        7136 obs  [1998-01-20 → 2026-06-02]
  ✓ VTI            Vanguard Total Stock Market ETF           6277 obs  [2001-06-15 → 2026-06-02]
  ✓ VT             Vanguard Total World ETF                  4511 obs  [2008-06-26 → 2026-06-02]
  ✓ EFA            iShares MSCI EAFE ETF                     6227 obs  [2001-08-27 → 2026-06-02]
  ✓ EEM            iShares MSCI Emerging Markets ETF         5821 obs  [2003-04-14 → 2026-06-02]
  ✓ VEA            Vanguard Developed Markets ETF            4743 obs  [2007

$^RVX: possibly delisted; no timezone found

1 Failed download:
['^RVX']: possibly delisted; no timezone found


  ✓ ^VIX           CBOE VIX                                  9173 obs  [1990-01-02 → 2026-06-03]
  ✓ ^VXN           Nasdaq-100 VIX                            6377 obs  [2001-01-23 → 2026-06-02]
  ✓ ^OVX           Crude Oil VIX                             4796 obs  [2007-05-10 → 2026-06-02]
  ✓ ^GVZ           Gold VIX                                  4528 obs  [2008-06-03 → 2026-06-02]
  ✓ ^EVZ           EuroCurrency VIX                          4174 obs  [2008-08-01 → 2025-03-05]
  ✓ ^VVIX          VIX of VIX                                4875 obs  [2007-01-03 → 2026-06-02]
  ✓ ^SKEW          CBOE Skew                                 9097 obs  [1990-01-02 → 2026-06-02]
  ✓ UVXY           ProShares Ultra VIX STF                   3686 obs  [2011-10-04 → 2026-06-02]
  ✓ SVXY           ProShares Short VIX STF                   3686 obs  [2011-10-04 → 2026-06-02]
  ✓ VXX            iPath S&P 500 VIX STF                     2099 obs  [2018-01-25 → 2026-06-02]
  ✓ VIXY           ProShares V

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TRYYUSD=X"}}}
$TRYYUSD=X: possibly delisted; no timezone found

1 Failed download:
['TRYYUSD=X']: possibly delisted; no timezone found


  ✓ EURUSD=X       EUR/USD                                   5839 obs  [2003-12-01 → 2026-06-03]
  ✓ GBPUSD=X       GBP/USD                                   5851 obs  [2003-12-01 → 2026-06-03]
  ✓ JPYUSD=X       JPY/USD                                   7673 obs  [1996-10-30 → 2026-06-03]
  ✓ CHFUSD=X       CHF/USD                                   5905 obs  [2003-09-17 → 2026-06-03]
  ✓ CADUSD=X       CAD/USD                                   5907 obs  [2003-09-17 → 2026-06-03]
  ✓ AUDUSD=X       AUD/USD                                   5215 obs  [2006-05-16 → 2026-06-03]
  ✓ NZDUSD=X       NZD/USD                                   5840 obs  [2003-12-01 → 2026-06-03]
  ✓ SGDUSD=X       SGD/USD                                   5851 obs  [2003-12-01 → 2026-06-03]
  ✓ CNYUSD=X       CNY/USD                                   6241 obs  [2001-06-25 → 2026-06-03]
  ✓ INRUSD=X       INR/USD                                   5836 obs  [2003-12-01 → 2026-06-03]
  ✓ BRLUSD=X       BRL/USD    